# YOLOv3 完整项目流程总结

## 一、项目概述

从零实现 YOLOv3 目标检测，在 `little_data`（4 类：人、猫、狗、马，18 张图片）上完成训练和推理。

## 二、完整流程

### 第 1 步：理论准备

| 概念 | 说明 |
|------|------|
| **YOLOv3 三部分损失** | 坐标损失 (MSE) + 置信度损失 (BCE) + 分类损失 (BCE) |
| **正负样本不平衡** | 每个图片 10647 个 anchor box，仅 ~9 个正样本 → 需要 λ_coord 和 λ_noobj 平衡 |
| **Anchor 机制** | 9 个预定义 anchor，分 3 组对应 3 个尺度 (13×13, 26×26, 52×52) |
| **NMS 后处理** | 从大量重叠候选框中为每个目标保留置信度最高的一个 |

### 第 2 步：标签编码

**输入**: `Parse_label.txt` → `cls cx cy w h`（像素坐标）

**输出**: 3 个尺度的标签张量 `[13,13,3,9]`, `[26,26,3,9]`, `[52,52,3,9]`

### 第 3 步：网络结构

```
输入 (3, 416, 416)
  └─ Darknet-53 骨干网络
       ├─ Stage 3 out → 52×52×256
       ├─ Stage 4 out → 26×26×512
       └─ Stage 5 out → 13×13×1024
  └─ FPN 特征金字塔
       ├─ 13×13 → ConvSet → 预测头 → out_13
       ├─ 上采样 → 与 26×26 拼接 → ConvSet → 预测头 → out_26
       └─ 上采样 → 与 52×52 拼接 → ConvSet → 预测头 → out_52
```

### 第 4 步：训练

```
优化器: Adam (lr=0.001, weight_decay=0.0005)
调度器: MultiStepLR [80, 150] gamma=0.1
损失:   5×pos(MSE+BCE+CE) + 0.5×neg(BCE)  [reduction='mean']
增强:   Mosaic + 随机水平翻转
Epochs: 200, Batch: 16
```

### 第 5 步：推理

```
模型前向 → 3 个尺度原始输出
  └─ decode_scale: sigmoid(conf), offset→cx, anchor_w×exp(tw)→bw
  └─ decode_all: 合并 10647 个候选框
  └─ NMS: 过滤 conf<0.1, 抑制 IoU>0.4 的同类框
  └─ 坐标还原: 416×416 → 原图尺寸
  └─ PIL 绘制矩形框
```

## 三、踩坑记录 & 修复

### Bug 1：分类损失 BCE(0,0) ≠ 0
- 掩码方式导致背景贡献 ~29513 无效损失
- 修复: boolean indexing 只取正样本

### Bug 2：置信度损失梯度冲突
- 正样本同时被推往 1 和 0
- 修复: 正负样本分开计算

### Bug 3：正负样本极度不平衡
- ~9 正 vs ~10638 负 → 梯度被负样本淹没
- 修复: Adam + reduction='mean' + 5:0.5 权重

## 四、YOLOv3 核心公式

### 损失函数
$$L = 5\sum\mathbb{1}^{obj}\text{MSE}(t,\hat{t}) + 5\sum\mathbb{1}^{obj}\text{BCE}(C,\hat{C}) + 0.5\sum\mathbb{1}^{noobj}\text{BCE}(C,\hat{C}) + 5\sum\mathbb{1}^{obj}\sum_c\text{CE}(p_c,\hat{p}_c)$$

### 解码公式
$$b_x = t_x + g_x \quad b_y = t_y + g_y \quad b_w = a_w e^{t_w} \quad b_h = a_h e^{t_h}$$
（注意：训练时 MSE 直接作用在 raw t_x, t_y 上，推理时不加 sigmoid）


In [ ]:
# -------------------- 推理 + 画框 --------------------
import torch, os, math
from tqdm import tqdm
from torchvision import transforms
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

from train import YOLOv3, DEVICE, IMG_SIZE, NUM_CLASSES, CLASSES, ANCHORS_GROUPS

CONF_THRESH = 0.1  # 旧模型阈值（新模型训练后可改回 0.3~0.5）

def decode_scale(pred, anchors, feat_size):
    B, H, W, na, _ = pred.shape
    stride = IMG_SIZE / H
    pred[...,0] = torch.sigmoid(pred[...,0])  # conf 转概率
    # tx, ty 不加 sigmoid！训练时 MSE(raw, offset) 无 sigmoid
    boxes = []
    for b in range(B):
        for i in range(H):
            for j in range(W):
                for a in range(na):
                    conf = pred[b,i,j,a,0].item()
                    if conf < CONF_THRESH: continue
                    tx, ty = pred[b,i,j,a,1].item(), pred[b,i,j,a,2].item()
                    cx, cy = (tx+j)*stride, (ty+i)*stride
                    bw = anchors[a][0] * math.exp(pred[b,i,j,a,3].item())
                    bh = anchors[a][1] * math.exp(pred[b,i,j,a,4].item())
                    x1, y1, x2, y2 = cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2
                    cls_id = torch.argmax(pred[b,i,j,a,5:]).item()
                    boxes.append([x1,y1,x2,y2,conf,cls_id])
    return boxes

def decode_all(outputs):
    boxes = []
    for feat, out in zip([13,26,52], outputs):
        B = out.size(0)
        p = out.permute(0,2,3,1).reshape(B,feat,feat,3,5+NUM_CLASSES)
        boxes.extend(decode_scale(p, ANCHORS_GROUPS[feat], feat))
    return boxes

def nms(boxes, iou_thresh=0.4):
    if not boxes: return []
    boxes = sorted(boxes, key=lambda x:x[4], reverse=True)
    keep = []
    while boxes:
        b = boxes.pop(0)
        keep.append(b)
        boxes = [x for x in boxes if x[5]!=b[5] or _iou(b[:4],x[:4])<iou_thresh]
    return keep

def _iou(b1, b2):
    x1,y1 = max(b1[0],b2[0]), max(b1[1],b2[1])
    x2,y2 = min(b1[2],b2[2]), min(b1[3],b2[3])
    inter = max(0,x2-x1)*max(0,y2-y1)
    return inter/((b1[2]-b1[0])*(b1[3]-b1[1])+(b2[2]-b2[0])*(b2[3]-b2[1])-inter+1e-6)

def letterbox(img, sz=416, color=(114,114,114)):
    w,h = img.size
    r = min(sz/w, sz/h)
    img = img.resize((int(w*r),int(h*r)), Image.BICUBIC)
    dw, dh = (sz-int(w*r))/2, (sz-int(h*r))/2
    c = Image.new('RGB', (sz,sz), color)
    c.paste(img, (int(dw),int(dh)))
    return c, r, (dw,dh)

def detect_and_draw():
    model = YOLOv3().to(DEVICE)
    ckpt = torch.load('checkpoints/yolov3_best.pth', map_location=DEVICE)
    sd = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(sd); model.eval()
    print(f'模型加载完成, 设备: {DEVICE}, 阈值: {CONF_THRESH}')

    img_dir, out_dir = 'little_data/images', 'little_data/outputs_detection'
    os.makedirs(out_dir, exist_ok=True)
    files = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg','.png','.jpeg'))])
    colors = [(255,0,0),(0,255,0),(0,0,255),(255,255,0)]
    tfm = transforms.ToTensor()

    for name in tqdm(files, desc='检测中'):
        img = Image.open(os.path.join(img_dir, name)).convert('RGB')
        ow, oh = img.size
        rs, r, (dw,dh) = letterbox(img)
        x = tfm(rs).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            o13, o26, o52 = model(x)
        boxes = nms(decode_all([o13,o26,o52]))
        draw = ImageDraw.Draw(img)
        for x1,y1,x2,y2,cf,ci in boxes:
            x1,y1 = max(0,(x1-dw)/r), max(0,(y1-dh)/r)
            x2,y2 = min(ow,(x2-dw)/r), min(oh,(y2-dh)/r)
            c = colors[int(ci)%len(colors)]
            draw.rectangle([x1,y1,x2,y2], outline=c, width=3)
            draw.text((x1,y1-15), f'{CLASSES[int(ci)]} {cf:.2f}', fill=c)
        img.save(os.path.join(out_dir, name))

    print(f'完成! {len(files)} 张, 结果: {out_dir}/')
    print('\n===== 预览 =====')
    for name in files[:9]:
        display(Image.open(os.path.join(out_dir, name)))
        print(name)

if __name__ == '__main__':
    detect_and_draw()


# yolov3_bug 代码 Bug 修复总结

## 📁 `dataset.py` — 3处修复
| # | 问题 | 修复 |
|---|------|------|
| ① | `math.modf` 解包错误 | 交换赋值 |
| ② | 返回 `label[52]` 两次，缺 `label[26]` | → `label[13],label[26],label[52]` |
| ③ | 变量名拼写 `tr_tranform` | → `tr_transform` |

## 📁 `yolov3.py` — 5处修复
| # | 问题 | 修复 |
|---|------|------|
| ① | `out_neck_13` 通道不匹配 | → `CBL(512,1024,3,1)` |
| ② | `out_neck_26` 通道不匹配 + 缺少 cls | → `CBL(256,512)` + `3*(5+CLS_NUMS)` |
| ③ | `contate_26` dim=0 → batch翻倍 | → dim=1 |
| ④ | `concate_52` 拼接了最终输出 | → 拼接 backbone 的 `out_52` |
| ⑤ | `CBLSET(128,128)` 输入应为384 | → `CBLSET(256+128,128)` |

## 📁 `train.py` — 6处修复
| # | 问题 | 修复 |
|---|------|------|
| ① | reshape 未先 permute | → `permute(0,2,3,1).reshape(...)` |
| ② | `target[:,1]` vs `out[:,1:5]` 维度不匹配 | → `target[:,1:5]` |
| ③ | one-hot 传入 CrossEntropyLoss | → `argmax(dim=1)` |
| ④ | `out_noobj = target[...]` 致命 | → `out[target_noobj_mask]` |
| ⑤ | 负样本损失对全部9通道计算 | → 只对置信度通道 |
| ⑥ | `self.loss.zero_grad()` 不存在 | → `self.opt.zero_grad()` |

## 📁 `detect.py` — 3处修复
| # | 问题 | 修复 |
|---|------|------|
| ① | reshape 写死 4 个 anchor | → 3 |
| ② | stride 硬编码为13 | → `IMG_SIZE/feature_size` |
| ③ | cx/cy 索引交换 | → cx用W索引，cy用H索引 |

## 📁 `trainv2.py` — 服务器训练修复
| # | 问题 | 修复 |
|---|------|------|
| ① | `reduction='sum'` 负样本碾压正样本 | → `reduction='mean'` |
| ② | `neg_loss *= 0.01` 极端降权 | → 权重 `5.0*pos + 0.5*neg` |
| ③ | 多余的 `total_loss / B` | → 直接 `return total_loss` |
| ④ | Mosaic 未启用 | → `use_mosaic=True` |
